# Baseline Transfer Learning Model for Error Classification

This notebook implements a baseline Computer Vision model using transfer learning to classify different types of application errors.

## 1. Import Libraries

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")

ModuleNotFoundError: No module named 'numpy'

## 2. Data Exploration

In [ ]:
# Define data path
data_dir = r"C:\Users\ibf\Desktop\TFM\Nou projecte\Data"

# Get all error categories
error_categories = os.listdir(data_dir)
error_categories = [cat for cat in error_categories if os.path.isdir(os.path.join(data_dir, cat))]
error_categories.sort()

print(f"Error Categories: {error_categories}")
print(f"Number of categories: {len(error_categories)}")

In [ ]:
# Count images per category
image_counts = {}
image_extensions = ('.png', '.jpg', '.jpeg')

for category in error_categories:
    cat_path = os.path.join(data_dir, category)
    images = [f for f in os.listdir(cat_path) 
              if f.lower().endswith(image_extensions)]
    image_counts[category] = len(images)
    print(f"{category}: {len(images)} images")

total_images = sum(image_counts.values())
print(f"\nTotal images: {total_images}")

In [ ]:
# Visualize data distribution
fig, ax = plt.subplots(figsize=(12, 6))
categories = list(image_counts.keys())
counts = list(image_counts.values())

bars = ax.bar(range(len(categories)), counts, color='steelblue')
ax.set_xticks(range(len(categories)))
ax.set_xticklabels(categories, rotation=45, ha='right')
ax.set_ylabel('Number of Images', fontsize=12)
ax.set_title('Image Distribution by Error Category', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

# Create summary dataframe
summary_df = pd.DataFrame({
    'Category': categories,
    'Image Count': counts,
    'Percentage': [f"{100*count/total_images:.1f}%" for count in counts]
})
print("\nData Summary:")
print(summary_df)

## 3. Data Preparation

In [ ]:
# Define hyperparameters
IMAGE_SIZE = (224, 224)  # Standard size for transfer learning models
BATCH_SIZE = 32
VALIDATION_SPLIT = 0.2
TEST_SPLIT = 0.1

print(f"Image Size: {IMAGE_SIZE}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Validation Split: {VALIDATION_SPLIT}")
print(f"Test Split: {TEST_SPLIT}")

In [ ]:
# Load datasets using Keras utility
# This automatically labels images based on directory names

# Load full dataset
dataset = image_dataset_from_directory(
    data_dir,
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=True,
    validation_split=VALIDATION_SPLIT,
    subset=None
)

# Split into training and validation
train_dataset = image_dataset_from_directory(
    data_dir,
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=True,
    validation_split=VALIDATION_SPLIT,
    subset='training'
)

val_dataset = image_dataset_from_directory(
    data_dir,
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=True,
    validation_split=VALIDATION_SPLIT,
    subset='validation'
)

class_names = train_dataset.class_names
num_classes = len(class_names)

print(f"Number of classes: {num_classes}")
print(f"Class names: {class_names}")
print(f"\nTraining batches: {tf.data.experimental.cardinality(train_dataset)}")
print(f"Validation batches: {tf.data.experimental.cardinality(val_dataset)}")

In [ ]:
# Normalize pixel values to [0, 1]
def normalize_img(image, label):
    return image / 255.0, label

train_dataset = train_dataset.map(normalize_img)
val_dataset = val_dataset.map(normalize_img)

# Optimize datasets for performance
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.prefetch(buffer_size=AUTOTUNE)

print("Datasets normalized and optimized.")

In [ ]:
# Visualize sample images
fig, axes = plt.subplots(2, 4, figsize=(14, 8))
axes = axes.flatten()

for images, labels in train_dataset.take(1):
    for idx in range(min(8, len(images))):
        ax = axes[idx]
        ax.imshow(images[idx])
        # Get class label
        class_idx = np.argmax(labels[idx])
        ax.set_title(class_names[class_idx], fontsize=10)
        ax.axis('off')

plt.suptitle('Sample Images from Different Error Categories', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Build Transfer Learning Model

In [ ]:
# Load pre-trained EfficientNetB0 model
base_model = tf.keras.applications.EfficientNetB0(
    input_shape=IMAGE_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)

# Freeze base model layers for transfer learning
base_model.trainable = False

print(f"Base model: EfficientNetB0")
print(f"Base model parameters: {base_model.count_params():,}")

In [ ]:
# Build the complete model
model = Sequential([
    tf.keras.applications.efficientnet.EfficientNetB0(
        input_shape=IMAGE_SIZE + (3,),
        include_top=False,
        weights='imagenet'
    ),
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
], name='ErrorClassificationModel')

# Freeze the base model
model.layers[0].trainable = False

print("Model Architecture:")
model.summary()

In [ ]:
# Compile the model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled successfully.")

## 5. Train Model

In [ ]:
# Define callbacks
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

print("Callbacks configured.")

In [ ]:
# Train the model
EPOCHS = 30

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Model Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTraining Summary:")
print(f"Total epochs: {len(history.history['loss'])}")
print(f"Best val accuracy: {max(history.history['val_accuracy']):.4f}")
print(f"Best val loss: {min(history.history['val_loss']):.4f}")

## 6. Model Evaluation

In [ ]:
# Evaluate on validation set
val_loss, val_accuracy = model.evaluate(val_dataset, verbose=0)

print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")

In [ ]:
# Get predictions for confusion matrix and classification report
y_pred = []
y_true = []

for images, labels in val_dataset:
    predictions = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(predictions, axis=1))
    y_true.extend(np.argmax(labels, axis=1))

y_pred = np.array(y_pred)
y_true = np.array(y_true)

print(f"Predictions shape: {y_pred.shape}")
print(f"True labels shape: {y_true.shape}")

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title('Confusion Matrix - Validation Set', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Classification Report
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# Per-class accuracy
per_class_acc = cm.diagonal() / cm.sum(axis=1)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(range(len(class_names)), per_class_acc, color='steelblue')
ax.set_xticks(range(len(class_names)))
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_ylabel('Recall/Accuracy', fontsize=12)
ax.set_title('Per-Class Accuracy (Recall)', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3)
ax.axhline(y=np.mean(per_class_acc), color='r', linestyle='--', label=f'Average: {np.mean(per_class_acc):.3f}')
ax.legend()

# Add value labels
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 7. Save Model

In [ ]:
# Create Models directory if it doesn't exist
models_dir = r"C:\Users\ibf\Desktop\TFM\Nou projecte\TFM\Models"
os.makedirs(models_dir, exist_ok=True)

# Save model
model_path = os.path.join(models_dir, 'baseline_model.h5')
model.save(model_path)
print(f"Model saved to: {model_path}")

# Save class names
import json
class_names_path = os.path.join(models_dir, 'class_names.json')
with open(class_names_path, 'w') as f:
    json.dump(class_names, f)
print(f"Class names saved to: {class_names_path}")

## 8. Model Summary and Key Insights

In [ ]:
# Summary statistics
summary_stats = {
    'Total Images': total_images,
    'Number of Classes': num_classes,
    'Training Samples': int(total_images * (1 - VALIDATION_SPLIT)),
    'Validation Samples': int(total_images * VALIDATION_SPLIT),
    'Image Size': str(IMAGE_SIZE),
    'Base Model': 'EfficientNetB0',
    'Total Parameters': model.count_params(),
    'Trainable Parameters': sum([np.prod(w.shape) for w in model.trainable_weights]),
    'Epochs Trained': len(history.history['loss']),
    'Final Train Accuracy': f"{history.history['accuracy'][-1]:.4f}",
    'Final Val Accuracy': f"{history.history['val_accuracy'][-1]:.4f}",
    'Best Val Accuracy': f"{max(history.history['val_accuracy']):.4f}",
    'Final Train Loss': f"{history.history['loss'][-1]:.4f}",
    'Final Val Loss': f"{history.history['val_loss'][-1]:.4f}",
}

print("\n" + "="*50)
print("BASELINE MODEL SUMMARY")
print("="*50)
for key, value in summary_stats.items():
    print(f"{key:.<30} {value}")
print("="*50)

In [ ]:
# Create a summary report
report_df = pd.DataFrame({
    'Error Category': class_names,
    'Train Samples': [image_counts[cat] for cat in class_names],
    'Recall': [f"{per_class_acc[i]:.4f}" for i in range(len(class_names))],
    'Support': cm.sum(axis=1)
})

print("\nPer-Class Performance:")
print(report_df.to_string(index=False))